In [9]:
# import os
# for f in ['train_binned.parquet', 'test_binned.parquet']:
#     p = f'data/processed/{f}'
#     if os.path.exists(p):
#         os.remove(p)
#         print(f"removed {f}")

# 02 — Feature Engineering, WOE & IV

Continues from `01_setup_and_data.ipynb`. **Runs top to bottom with Run All.**

Phase 1 established: target = Charged Off/Default vs Fully Paid, unresolved loans
dropped, censoring bias diagnosed by resolution rate per vintage, time-based split
train 2007–2014 (453,809 / bad 0.170) vs test 2015 (375,546 / bad 0.202).

This notebook applies the leakage screen, groups categoricals, bins continuous
features, computes WOE/IV, and saves processed frames to `data/processed/`.

**Block A caches to parquet.** First run reads the 1GB CSV in chunks; every later
run loads parquet in seconds. Delete `data/processed/*.parquet` to force a rebuild.

## Setup

In [10]:
# ============================================================
# BLOCK 0 — imports, constants, temp dir off C:
# ============================================================
import os, json, gc
import pandas as pd
import numpy as np

os.environ['TMP'] = os.environ['TEMP'] = r'X:\Projects\credit-pd-engine\tmp'
os.makedirs(os.environ['TMP'], exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

DATA = 'data/accepted_2007_to_2018Q4.csv.gz'
CACHE_TRAIN = 'data/processed/train_binned.parquet'
CACHE_TEST  = 'data/processed/test_binned.parquet'

good_statuses = ['Fully Paid',
                 'Does not meet the credit policy. Status:Fully Paid']
bad_statuses  = ['Charged Off',
                 'Does not meet the credit policy. Status:Charged Off',
                 'Default']

pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 130)
print(f"pandas {pd.__version__} | numpy {np.__version__}")

pandas 2.3.3 | numpy 2.4.6


In [11]:
# ============================================================
# BLOCK 1 — WOE / IV helper
# WOE = ln(%good / %bad). Positive = safer than portfolio average.
# dropna=False keeps missing as its own bin — missingness is informative.
# Laplace smoothing prevents infinite WOE when a bin has zero bads.
# ============================================================
def woe_iv(data, feature, target='target', laplace=0.5):
    grp = data.groupby(feature, dropna=False, observed=False)[target].agg(
        n='size', bad='sum')
    grp['good'] = grp['n'] - grp['bad']
    G, B, k = grp['good'].sum(), grp['bad'].sum(), len(grp)

    grp['pct_good'] = (grp['good'] + laplace) / (G + laplace * k)
    grp['pct_bad']  = (grp['bad']  + laplace) / (B + laplace * k)
    grp['woe'] = np.log(grp['pct_good'] / grp['pct_bad'])
    grp['iv_contrib'] = (grp['pct_good'] - grp['pct_bad']) * grp['woe']
    grp['bad_rate']   = grp['bad'] / grp['n']
    grp['pct_of_pop'] = grp['n'] / grp['n'].sum()

    return (grp[['n', 'pct_of_pop', 'bad_rate', 'woe', 'iv_contrib']],
            grp['iv_contrib'].sum())

## Block A — load

**Leakage rule:** would the underwriter have had this number the day they approved
the loan? Excluded accordingly: `total_pymnt`, `out_prncp`, `last_pymnt_*`,
`recoveries`, `collection_recovery_fee`, all `hardship_*` / `settlement_*`, and —
the subtle one — `last_fico_range_*`, which is the FICO at the most recent pull,
i.e. *after* a defaulting borrower started missing payments.

`int_rate` / `grade` / `sub_grade` load but never enter the model. They pass the
leakage test but are another model's output; they serve as the benchmark in nb 03.

Chunked read with `float32` — the full-file load blew up memory previously.

In [12]:
# ============================================================
# BLOCK A — load feature set, rebuild split (cached)
# ============================================================
feature_cols = [
    'loan_amnt', 'term', 'installment',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    'purpose', 'addr_state', 'dti',
    'earliest_cr_line', 'fico_range_low', 'fico_range_high',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'mort_acc',
    'pub_rec_bankruptcies', 'application_type', 'initial_list_status',
]
lc_view = ['int_rate', 'grade', 'sub_grade']
keys    = ['id', 'issue_d', 'loan_status']

CACHED = os.path.exists(CACHE_TRAIN) and os.path.exists(CACHE_TEST)

if CACHED:
    train = pd.read_parquet(CACHE_TRAIN)
    test  = pd.read_parquet(CACHE_TEST)
    print("Loaded from parquet cache — CSV skipped.")
else:
    dtypes = {c: 'float32' for c in [
        'loan_amnt','installment','annual_inc','dti','fico_range_low',
        'fico_range_high','delinq_2yrs','inq_last_6mths','open_acc','pub_rec',
        'revol_bal','revol_util','total_acc','mort_acc','pub_rec_bankruptcies',
        'int_rate']}
    dtypes['id'] = 'str'

    parts = []
    for ch in pd.read_csv(DATA, compression='gzip',
                          usecols=keys + feature_cols + lc_view,
                          dtype=dtypes, chunksize=200_000):
        parts.append(ch[ch['loan_status'].isin(good_statuses + bad_statuses)])
    df = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()

    df['target']  = df['loan_status'].isin(bad_statuses).astype('int8')
    df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
    df['vintage'] = df['issue_d'].dt.year

    train = df[df['vintage'].between(2007, 2014)].copy()
    test  = df[df['vintage'] == 2015].copy()
    del df; gc.collect()

    assert len(train) == 453_809, f"train {len(train):,} != 453,809"
    assert len(test)  == 375_546, f"test {len(test):,} != 375,546"
    print("Rebuilt from CSV — matches notebook 01.")

print(f"Train: {len(train):,} | bad rate = {train['target'].mean():.3f}")
print(f"Test:  {len(test):,} | bad rate = {test['target'].mean():.3f}")

Rebuilt from CSV — matches notebook 01.
Train: 453,809 | bad rate = 0.170
Test:  375,546 | bad rate = 0.202


## Block B — screens

Three checks, all run on **train only** where they inform feature decisions.
Deciding anything from test behaviour is feature selection on the test set.

In [13]:
# ============================================================
# BLOCK B — missingness, vintage artefacts, category alignment
# ============================================================
if not CACHED:
    miss = pd.DataFrame({
        'train_null_%': train[feature_cols].isna().mean().mul(100).round(2),
        'test_null_%':  test[feature_cols].isna().mean().mul(100).round(2),
    })
    miss['gap'] = (miss['train_null_%'] - miss['test_null_%']).abs().round(2)
    print(miss.sort_values('train_null_%', ascending=False).head(8), "\n")

    print("mort_acc null % by vintage:")
    print(train.groupby('vintage')['mort_acc'].apply(
        lambda s: round(s.isna().mean()*100, 2)).to_string(), "\n")

    for c in ['purpose','home_ownership','verification_status',
              'application_type','initial_list_status','term','emp_length']:
        tr, te = set(train[c].dropna().unique()), set(test[c].dropna().unique())
        msg = f"{c:22s} train={len(tr):3d} test={len(te):3d}"
        if tr - te: msg += f"  | train-only: {sorted(tr - te)}"
        if te - tr: msg += f"  | TEST-ONLY: {sorted(te - tr)}"
        print(msg)
else:
    print("Cached — screens already applied.")

                      train_null_%  test_null_%    gap
mort_acc                     11.02         0.00  11.02
emp_length                    4.53         5.87   1.34
pub_rec_bankruptcies          0.30         0.00   0.30
revol_util                    0.07         0.04   0.03
earliest_cr_line              0.01         0.00   0.01
inq_last_6mths                0.01         0.00   0.01
open_acc                      0.01         0.00   0.01
pub_rec                       0.01         0.00   0.01 

mort_acc null % by vintage:
vintage
2007    100.00
2008    100.00
2009    100.00
2010    100.00
2011    100.00
2012     14.04
2013      0.00
2014      0.00 

purpose                train= 14 test= 14
home_ownership         train=  6 test=  4  | train-only: ['NONE', 'OTHER']
verification_status    train=  3 test=  3
application_type       train=  1 test=  2  | TEST-ONLY: ['Joint App']
initial_list_status    train=  2 test=  2
term                   train=  2 test=  2
emp_length             train= 11

### Findings

- **`mort_acc`** 100% null 2007–2011, 14% in 2012, 0% after — the field didn't
  exist before mid-2012. "Null" means "issued pre-2012", so its WOE would encode
  origination year. **Drop.**
- **`application_type`** has one level in train and a TEST-ONLY `Joint App` in
  2015 — no variance to learn from, and a level with no WOE to apply. **Drop.**
  (Joint applications are a genuinely different risk product with zero training
  exposure — a real production limitation, worth stating.)
- **`fico_range_high`** = `fico_range_low` + 4 mechanically. Perfectly collinear.
  **Drop.**
- **`emp_length`** nulls rise 0% → 5.2% gradually, so *not* a vintage proxy.
  Real behaviour: people who leave it blank default at 22.1% vs 16.8%. Keep as
  a binary — the 11 tenure levels span only 1.4pp of bad rate and are noise.
- **`home_ownership`** `OTHER`/`NONE`/`ANY` = 233 rows. Too thin to estimate WOE,
  so group by *economic logic* rather than observed rate: all mean "neither
  mortgaged nor owned" → fold into `RENT`.

In [14]:
# ============================================================
# BLOCK C — drop decisions, groupings, binning
# ============================================================
DROP_LEAK_OR_ARTEFACT = ['application_type', 'initial_list_status',
                         'fico_range_high', 'mort_acc']
feature_cols = [c for c in feature_cols if c not in DROP_LEAK_OR_ARTEFACT]

if not CACHED:
    # 29 rows with a failed bureau pull — all bureau fields null together
    train = train[train['open_acc'].notna()].copy()
    test  = test[test['open_acc'].notna()].copy()

purpose_groups = {
    'small_business': 'small_business',          # kept alone — worst in all 8 vintages
    'debt_consolidation': 'debt_consolidation',
    'credit_card': 'credit_card',
    'home_improvement': 'home_improvement',
    'other': 'other_risky', 'medical': 'other_risky', 'house': 'other_risky',
    'moving': 'other_risky', 'renewable_energy': 'other_risky',
    'educational': 'other_risky',                # discontinued product
    'car': 'purchase_safe', 'major_purchase': 'purchase_safe',
    'wedding': 'purchase_safe', 'vacation': 'purchase_safe',
}
ho_groups  = {'MORTGAGE':'MORTGAGE', 'RENT':'RENT', 'OWN':'OWN',
              'OTHER':'RENT', 'NONE':'RENT', 'ANY':'RENT'}
ver_groups = {'Verified':'Verified', 'Source Verified':'Verified',
              'Not Verified':'Not Verified'}

# Edges on round numbers near the quantiles — a cutpoint at $55,000 is
# explicable to a credit committee; one at $54,317 was fitted to the data.
FICO_EDGES = [-np.inf, 660, 675, 690, 710, 740, np.inf]
BIN_EDGES = {
    'dti':            [-np.inf, 10, 15, 20, 25, 30, np.inf],
    'revol_util':     [-np.inf, 25, 40, 55, 70, 85, np.inf],
    'annual_inc':     [-np.inf, 40_000, 55_000, 75_000, 100_000, 150_000, np.inf],
    'loan_amnt':      [-np.inf, 7_000, 11_000, 15_000, 20_000, 28_000, np.inf],
    'inq_last_6mths': [-np.inf, 0, 1, 2, np.inf],
}

if not CACHED:
    inc_median = train['annual_inc'].median()   # TRAIN median applied to both

    for d in (train, test):
        d['purpose_grp']        = d['purpose'].map(purpose_groups)
        d['home_ownership_grp'] = d['home_ownership'].map(ho_groups)
        d['verif_grp']          = d['verification_status'].map(ver_groups)
        d['emp_length_stated']  = d['emp_length'].notna().map(
                                      {True: 'stated', False: 'not_stated'})
        d['revol_util'] = d['revol_util'].fillna(100)   # missing ≈ no revolving line
        d['annual_inc'] = d['annual_inc'].fillna(inc_median)

        d['fico_bin'] = pd.cut(d['fico_range_low'], bins=FICO_EDGES)
        for col, edges in BIN_EDGES.items():
            d[f'{col}_bin'] = pd.cut(d[col], bins=edges)

FINAL_BINS = ['term', 'fico_bin', 'dti_bin', 'verif_grp', 'annual_inc_bin',
              'revol_util_bin', 'loan_amnt_bin', 'inq_last_6mths_bin',
              'purpose_grp', 'home_ownership_grp', 'emp_length_stated']

assert len(train) == 453_780 and len(test) == 375_546
print(f"Train {len(train):,} | Test {len(test):,}")
print("Missing bins:", [c for c in FINAL_BINS if c not in train.columns] or "none")

Train 453,780 | Test 375,546
Missing bins: none


## Block D — WOE / IV tables

Monotonicity check on ordered features: `fico_bin` and `dti_bin` must move in one
direction. A scorecard where a *higher* FICO scores *worse* in one bucket is
indefensible to a credit committee even if the wiggle is real in-sample.

In [15]:
# ============================================================
# BLOCK D — WOE tables and IV ranking
# ============================================================
iv_summary = {}
for f in FINAL_BINS:
    tbl, iv = woe_iv(train, f)
    iv_summary[f] = iv
    print(f"\n=== {f} | IV = {iv:.4f} ===")
    print(tbl.round(4).to_string())

print("\n\n--- IV ranking ---")
print(pd.Series(iv_summary).sort_values(ascending=False).round(4).to_string())
print("\nSiddiqi: <0.02 drop | 0.02-0.1 weak | 0.1-0.3 medium | >0.5 SUSPECT LEAKAGE")


=== term | IV = 0.1954 ===
                n  pct_of_pop  bad_rate     woe  iv_contrib
term                                                       
36 months  337967      0.7448    0.1316  0.3033      0.0618
60 months  115813      0.2552    0.2831 -0.6548      0.1335

=== fico_bin | IV = 0.1085 ===
                     n  pct_of_pop  bad_rate     woe  iv_contrib
fico_bin                                                        
(-inf, 660.0]    37271      0.0821    0.2292 -0.3708      0.0127
(660.0, 675.0]  111032      0.2447    0.2106 -0.2623      0.0183
(675.0, 690.0]   98867      0.2179    0.1878 -0.1192      0.0032
(690.0, 710.0]   96547      0.2128    0.1568  0.0981      0.0020
(710.0, 740.0]   70431      0.1552    0.1216  0.3938      0.0211
(740.0, inf]     39632      0.0873    0.0775  0.8934      0.0512

=== dti_bin | IV = 0.0540 ===
                   n  pct_of_pop  bad_rate     woe  iv_contrib
dti_bin                                                       
(-inf, 10.0]   90062   

### Features dropped on IV grounds

Screened out below the 0.02 floor: `installment` (0.0162, and non-monotonic — the
top bin reverses because large installments belong to *36-month* loans, so it's
contaminated by term structure), `revol_bal` (0.0038, U-shaped: large balances are
*safer*, because balance without a limit is a credit-capacity proxy not a burden
measure), `open_acc` (0.0041), `total_acc` (0.0007), `pub_rec` (0.0021),
`delinq_2yrs` (0.0013), `pub_rec_bankruptcies` (0.0012), `cr_hist_yrs` (0.0070),
`addr_state` (0.0103 grouped).

The adverse-credit features are weak **in this dataset, not in reality** — LC
rejected serious derogatory records before origination, so only the mild tail
survives. Range restriction, same mechanism that caps FICO's IV at 0.109.

`addr_state` decomposed into three different things sharing one WOE: a passed
macro shock (NV runs 0.41 → 0.20 as the housing crash recedes), a genuine trend
(AL deteriorating), and differential sampling (TN absent in 2009/2010/2012 —
LC suspended lending there, so its pooled rate is computed from bad years only).

In [16]:
# ============================================================
# BLOCK E — save processed frames for notebooks 03 and 04
# Binned columns cast to str: pd.cut Categoricals don't survive parquet.
# Raw columns saved from the SAME frames, so both models see identical rows.
# ============================================================
if not CACHED:
    meta_cols = ['id', 'issue_d', 'vintage', 'target', 'grade', 'sub_grade', 'int_rate']
    raw_cols  = ['loan_amnt','term','installment','emp_length','home_ownership',
                 'annual_inc','verification_status','purpose','addr_state','dti',
                 'earliest_cr_line','fico_range_low','delinq_2yrs','inq_last_6mths',
                 'open_acc','pub_rec','revol_bal','revol_util','total_acc',
                 'pub_rec_bankruptcies']

    for d, binned, raw in [(train, CACHE_TRAIN, 'data/processed/train_raw.parquet'),
                           (test,  CACHE_TEST,  'data/processed/test_raw.parquet')]:
        out = d[meta_cols + FINAL_BINS].copy()
        for c in FINAL_BINS:
            out[c] = out[c].astype(str)
        out.to_parquet(binned, index=False)
        d[meta_cols + raw_cols].to_parquet(raw, index=False)

    with open('data/processed/feature_set.json', 'w') as f:
        json.dump({'final_bins': FINAL_BINS,
                   'fico_edges': [str(x) for x in FICO_EDGES],
                   'bin_edges': {k: [str(x) for x in v] for k, v in BIN_EDGES.items()},
                   'purpose_groups': purpose_groups,
                   'iv': {k: round(v, 4) for k, v in iv_summary.items()},
                   'train_rows': len(train), 'test_rows': len(test)}, f, indent=2)
    print("Saved binned + raw parquets and feature_set.json")
else:
    print("Cache already present — nothing re-saved.")

for f in sorted(os.listdir('data/processed')):
    print(f"  {f:32s} {os.path.getsize('data/processed/'+f)/1e6:7.1f} MB")

Saved binned + raw parquets and feature_set.json
  feature_set.json                     0.0 MB
  scorecard_coefficients.csv           0.0 MB
  scorecard_points.csv                 0.0 MB
  test_binned.parquet                 13.0 MB
  test_raw.parquet                    20.5 MB
  test_scored.parquet                  8.3 MB
  train_binned.parquet                15.6 MB
  train_raw.parquet                   24.6 MB
